In [26]:
import requests
import base64
import pandas as pd
from typing import Dict, Any
from dotenv import load_dotenv
load_dotenv()
import os
import json
from typing import Dict, Any

In [27]:
def folder_to_df_merged(folder_path: str) -> Dict[str, pd.DataFrame]:
    result = {}

    for file in os.listdir(folder_path):
        if not file.endswith(".json"):
            continue

        path = os.path.join(folder_path, file)

        with open(path, encoding="utf-8") as f:
            data: Any = json.load(f)

        if isinstance(data, list) and all(isinstance(i, dict) for i in data):
            df = pd.json_normalize(data)

        elif isinstance(data, dict):
            list_keys = [k for k, v in data.items() if isinstance(v, list)]

            if list_keys:
                main_key = list_keys[0]
                df = pd.json_normalize(
                    data,
                    record_path=main_key,
                    meta=[k for k in data.keys() if k != main_key],
                    errors="ignore"
                )
            else:
                df = pd.DataFrame([data])

        else:
            df = pd.DataFrame()

        result[file.replace(".json", "")] = df
    
    return pd.concat(result.values(), ignore_index=True)

df = folder_to_df_merged("../data/raw")
df_result = df.explode("artist_genres").dropna(subset=['artist_genres'])


In [28]:
df_result = df_result.replace({'True': 1, 'False': 0})

In [29]:
df_result = df_result.drop(columns = ['platform','conn_country', 'ip_addr','spotify_track_uri', 'episode_name',
       'episode_show_name', 'spotify_episode_uri', 'audiobook_title',
       'audiobook_uri', 'audiobook_chapter_uri', 'audiobook_chapter_title',"master_metadata_track_name", "master_metadata_album_artist_name", "master_metadata_album_album_name", "offline", "incognito_mode","offline_timestamp", "artist_id"])

In [30]:
df_result['ms_played'] = pd.to_numeric(df_result['ms_played'], errors='coerce')

In [54]:
# Cria um score, "penalizando" skip/forward
df_result["score"] = (
    df_result["ms_played"]
    * (1 - df_result["skipped"])
    * (df_result["reason_end"] != "forward")
)
df_result = df_result[df_result["score"] != 0]
df_result

,ts,ms_played,reason_start,reason_end,shuffle,skipped,user,artist_genres,hour,periodo,score
2,2023-11-03 19:08:49+00:00,114865,clickrow,trackdone,False,False,gigi,melodic rap,19,noite,114865
4,2023-11-03 19:12:12+00:00,90010,playbtn,unexpected-exit,False,False,gigi,melodic rap,19,noite,90010
5,2023-11-03 21:44:28+00:00,159129,appload,trackdone,False,False,gigi,melodic rap,21,noite,159129
6,2023-11-03 21:47:12+00:00,162546,trackdone,trackdone,False,False,gigi,melodic rap,21,noite,162546
7,2023-11-03 21:49:55+00:00,162546,trackdone,trackdone,False,False,gigi,melodic rap,21,noite,162546
...,...,...,...,...,...,...,...,...,...,...,...
319630,2025-12-30 02:45:34+00:00,141302,trackdone,trackdone,False,False,Fefo,uk grime,2,madrugada,141302
319631,2025-12-30 03:04:26+00:00,261303,trackdone,trackdone,False,False,Fefo,uk drill,3,madrugada,261303
319631,2025-12-30 03:04:26+00:00,261303,trackdone,trackdone,False,False,Fefo,grime,3,madrugada,261303
319631,2025-12-30 03:04:26+00:00,261303,trackdone,trackdone,False,False,Fefo,uk grime,3,madrugada,261303


In [89]:
pivot = (
    df_result.groupby(["user", "artist_genres"])
      .agg(
          score=("score", "mean")
      )
      .reset_index()
).pivot(
    index="user",
    columns="artist_genres",
    values="score"
).fillna(0)

In [90]:
pivot

artist_genres,acid jazz,acid rock,acid techno,acoustic pop,adult standards,afro house,afro r&b,afro soul,afro tech,afro-cuban jazz,...,vietnamese bolero,vietnamese lo-fi,visual kei,vocal jazz,vocaloid,west coast hip hop,witch house,worship,yacht rock,zouk
user,,,,,,,,,,,,,,,,,,,,,
Allanabre,0.0,0.0,0.00,292566.0,138290.571429,0.000000,28001.000000,0.000000,0.0,0.0,...,0.0,0.000000,2501.000000,138290.571429,239410.888889,146921.500000,193440.230769,232660.000000,109077.586207,0.0
Cury,216106.0,0.0,0.00,0.0,10868.000000,0.000000,0.000000,0.000000,0.0,0.0,...,4475.0,0.000000,0.000000,0.000000,0.000000,188481.800000,0.000000,0.000000,256022.000000,0.0
Fefo,0.0,24281.0,0.00,0.0,159925.393939,0.000000,124003.333333,95794.606061,0.0,157081.0,...,0.0,129554.666667,0.000000,177213.442857,210102.000000,180781.809524,112803.812500,23533.333333,150465.000000,0.0
Gueguelas,0.0,0.0,212758.00,93160.0,33191.400000,3054.500000,105402.142857,0.000000,0.0,0.0,...,0.0,0.000000,65390.272727,37976.294118,5446.666667,181255.603509,76321.642857,543.333333,38000.000000,0.0
gigi,0.0,0.0,93064.25,0.0,1258.000000,101582.428571,130591.000000,0.000000,61648.0,0.0,...,0.0,0.000000,0.000000,1258.000000,0.000000,196555.480687,92536.000000,0.000000,177595.428571,253228.0


In [91]:
from sklearn.neighbors import NearestNeighbors

knn = NearestNeighbors(
    metric="cosine",
    algorithm="brute"
)

knn.fit(pivot)

NearestNeighbors(algorithm='brute', metric='cosine')

In [109]:
distancias, indices = knn.kneighbors(
    pivot.loc[["gigi"]],
    n_neighbors=3  # com 6 usuários, 2 ou 3 é ideal
)

usuarios_parecidos = pivot.index[indices.flatten()][1:] 
similaridades = 1 - distancias.flatten()[1:] 

for u, s in zip(usuarios_parecidos, similaridades):
    print(u, round(s, 3))


Cury 0.529
Gueguelas 0.5


In [110]:
import numpy as np

sim_vizinhos = similaridades
matriz_vizinhos = pivot.loc[usuarios_parecidos]

media_ponderada = np.average(
    matriz_vizinhos,
    axis=0,
    weights=sim_vizinhos
)

media_ponderada = pd.Series(
    media_ponderada,
    index=pivot.columns
)

usuario_vector = pivot.loc["Fefo"]

recomendacao = (media_ponderada - usuario_vector)

recomendacao = recomendacao.sort_values(ascending=False)

In [111]:
top5 = recomendacao.head(5)
print(top5)

artist_genres
doom metal      325574.253630
stoner rock     324366.363526
thrash metal    317937.320267
art rock        241455.477260
heavy metal     232726.504005
dtype: float64
